# Haskell: making the bug unrepresentable

## The climax of the climb
Every rung up to now has been about *catching* bugs. Idiomatic C# made
the compiler emit `CS8509` on a non-exhaustive switch expression. F#
made `match` warn on a missing DU case and let you flip it to a build
error. Each rung shrank the set of bugs you could ship without the
compiler shouting.

Haskell asks a different question. Not "does the compiler warn me"
but **is there a value of this type that could even represent the
bug?**

If the type system can't express the bug, you cannot ship it. You
cannot write a test that hits it. You can't even write a unit test
that *demonstrates* it without the test itself failing to compile.
The footgun isn't avoided. It's structurally absent.

Let's walk the seven-footgun catalogue one more time, on the cliff
face.

## The seven footguns, Haskell column

Recall the catalogue from chapter 3:

1. NRE in the logger — `d.Vehicle.LicensePlate` when `Vehicle` is null
2. Case-sensitive mode string — `"retry"` vs `"Retry"`
3. Missing recovery branch — `AggressiveRecovery` falls through silently
4. Mutable response — caller appends to `Errors` after the engine returns
5. Status typos compile — `"Accepetd"` ships
6. Validator throws on first failure — only one error per row reported
7. `switch` without a default — new case silently dropped

### Footgun 1 — Rogue logger touches a nullable field

**Representable in Haskell?** No.

The classifier's type is `ValidationConfig -> UploadMode -> RowContext
-> RowDecision`. No `IO` anywhere. You cannot call `putStrLn`
(Haskell's `Console.WriteLine`) inside a function with that signature
— the compiler refuses with a type mismatch. To add a logger, you'd
have to change the signature to `… -> IO RowDecision`, which forces
every caller all the way up to `main` into `IO`. Logging cannot hide.

And even if it were allowed, there is no nullable field to crash on.
Each `RowDecision` constructor carries exactly its case's payload.
There's no shared shape with a "sometimes null `Vehicle`" hanging off
it. The crash site doesn't exist as a value.

### Footgun 2 — Case-sensitive mode string

**Representable in Haskell?** Inside the domain — no. At the
boundary — yes, but caught.

`UploadMode` is a closed sum type:

In [ ]:
data UploadMode
  = Normal
  | Retry
  | ConservativeRecovery
  | AggressiveRecovery
  deriving stock (Eq, Show)

Once you're holding an `UploadMode`, you cannot have an unknown
variant. The string `"retry"` cannot be implicitly compared. It's
just not a value of that type.

The string only exists at the boundary, in `Api.hs`:

In [ ]:
parseMode :: String -> String -> Either [FuelUploadMappingError] UploadMode
parseMode field value =
  case normalize value of
    "normal" -> Right Normal
    "retry" -> Right Retry
    "conservativerecovery" -> Right ConservativeRecovery
    "aggressiverecovery" -> Right AggressiveRecovery
    _ -> Left [FuelUploadMappingError InvalidUploadMode field ...]

The return type is `Either [FuelUploadMappingError] UploadMode`. The
caller cannot pretend an unparsed string is a valid mode. They have
to handle the `Left` branch — that's the type system speaking.

### Footgun 3 — Missing recovery branch

**Representable in Haskell?** No.

The recovery matrix in `DuplicatePolicy.hs` is a series of equations
over `(UploadMode, DuplicateState)`. Add a fifth `UploadMode` case
called `ManualOverride` to `Domain/Primitive.hs` and run `cabal
build`. You get:

In [ ]:
src/FuelUpload/DuplicatePolicy.hs: warning: [-Wincomplete-patterns]
    Pattern match(es) are non-exhaustive
    In an equation for 'duplicateDecision':
        Patterns of type 'UploadMode', 'DuplicateState' not matched:
            ManualOverride (DuplicateOf _)

This project's `.cabal` enables `-Wincomplete-uni-patterns`, and the
conventional production setting flips warnings to errors via
`-Werror`. The build fails until you've handled every combination.
You cannot ship the unhandled case.

C#'s `switch` *expression* has a similar property (CS8509), but its
`switch` *statement* does not — and the normal-C# version uses
statements. Haskell has only one `case` form, and it's always
checked.

### Footgun 4 — Mutable response

**Representable in Haskell?** No.

There is no `mutable` keyword in Haskell. There is no `var`. Records
cannot be mutated; the closest you get is record-update syntax, which
*creates a new value* that differs from the old one by the named
fields. The old value is untouched.

In [ ]:
bumpTotal summary =
  summary {summaryTotalRows = summaryTotalRows summary + 1}

That doesn't change `summary`. It returns a new `BatchSummary`. If a
caller holds the old `summary` after this call, they still hold the
old value. Aliasing-based mutation isn't possible because nothing
mutates.

To get true in-place mutability you need `IORef` or `STRef` — and
those types themselves appear in your function signature (`IORef`
lives in `IO`, `STRef` in `ST`). Hidden mutation is impossible.
Footgun 4 cannot be written in Haskell at all.

### Footgun 5 — Status typos compile

**Representable in Haskell?** No.

There are no status strings. The "status" of a row decision is the
constructor name:

In [ ]:
data RowDecision
  = Accepted FuelTransaction
  | AcceptedWithWarnings FuelTransaction (NonEmpty ValidationWarning)
  | Quarantined FuelTransaction (NonEmpty QuarantineReason)
  | SkippedDuplicate SkippedDuplicate
  | Rejected RejectedRow
  | Fatal FatalError
  deriving stock (Eq, Show)

Typing `Accepetd transaction` produces a compile error: "Data
constructor not in scope: `Accepetd`." There's no string layer for
the typo to survive into. The serialization to JSON happens at the
boundary, in `Api.hs`, where the constructor *maps* to a string —
but that mapping is itself one function, written once, and any new
constructor that lacks a string mapping is an exhaustiveness warning.

### Footgun 6 — Validator throws on first failure

**Representable in Haskell?** Yes, but the codebase doesn't.

You *could* throw via `error "bad row"`. The community treats that as
a code smell; the natural shape is to return a list:

In [ ]:
validationErrors :: ValidationConfig -> ParsedFuelRow -> [ValidationError]
validationErrors config row =
  quantityErrors <> amountErrors <> odometerErrors

`[ValidationError]` — a list of typed errors, accumulated. Each check
returns either `[]` or a single-element list; they concatenate. The
caller gets every error the row produced, not just the first.

Not enforced by the type system — you *could* throw — but the language
steers you the other way. Exceptions are clumsy; `Either e a` and
lists are easy.

### Footgun 7 — `switch` without a default

**Representable in Haskell?** No.

There is no "default-less switch" because there's only `case`, and
`case` is always checked by `-Wincomplete-uni-patterns`. Every
constructor must be handled or the build warns. With `-Werror`, the
build fails.

The `Summary.hs` `summarizeRows` function is the proof:

In [ ]:
addDecision decision summary =
  case decision of
    Accepted _ -> countAccepted summary
    AcceptedWithWarnings _ _ -> countAcceptedWithWarnings summary
    Quarantined _ _ -> countQuarantined summary
    SkippedDuplicate _ -> countSkipped summary
    Rejected _ -> countRejected summary
    Fatal _ -> countFatal summary

No default. No fallthrough. No `_ -> error "unknown"`. If a seventh
constructor were added, every `case` like this in the codebase would
warn, the build would fail, and the missing branches would be
enumerated by the compiler.

### Scorecard

| # | Footgun | Normal C# | Idiomatic C# | F# | Haskell |
|---|---|---|---|---|---|
| 1 | NRE in logger | possible | partial (NRT opt-in) | partial | **impossible** (IO not in signature) |
| 2 | Case-sensitive mode | possible | enum | DU | **closed sum type** |
| 3 | Missing recovery branch | possible | expression warn | match warn | `case` warn + Werror |
| 4 | Mutable response | possible | IReadOnlyList view | immutable default | **no mutability without IORef/IO** |
| 5 | Status typo | possible | sealed record per case | DU case | **constructor name** |
| 6 | Validator throws first | possible | accumulate | accumulate | accumulate (Either/list idiom) |
| 7 | Switch no default | possible | expression warn | match warn | `case` warn + Werror |

Seven for seven. Either the type cannot represent the bug at all, or
the compiler refuses the build when it can.

## The multiplier: property tests

Algebraic types and property tests are made for each other. The
constructors *are* the cases your test needs to cover; generating a
random `RowDecision` means picking a constructor uniformly and
filling its payload with random data. **QuickCheck** is the library
that does this.

A property test, briefly: instead of writing one example case
(`testThatTotalIsThreeWhenWeHaveThreeRows`), you state a property
that should hold for *any* input, and let QuickCheck generate ~100
random inputs and check the property on each. If any input
violates the property, QuickCheck shrinks it down to a minimal
counterexample and reports it.

Here's the property test for batch summary totals
(`test/FuelUpload/Properties.hs`):

In [ ]:
prop "summary total is the number of row decisions" \(Decisions decisions) ->
  summaryTotalRows (summarizeRows decisions) == length decisions

prop "summary count partitions always add up to total" \(Decisions decisions) ->
  let summary = summarizeRows decisions
   in summaryAccepted summary
        + summaryQuarantined summary
        + summarySkippedDuplicates summary
        + summaryRejected summary
        + summaryFatal summary
        == summaryTotalRows summary

Two statements, each backed by 100 random batches of decisions:

1. The summary total equals the number of decisions. Always.
2. The per-outcome counts partition the total. Always.

The `Arbitrary` instance generates random `RowDecision`s by picking a
constructor:

In [ ]:
arbitraryDecision :: Gen RowDecision
arbitraryDecision =
  oneof
    [ Accepted <$> arbitraryTransaction
    , AcceptedWithWarnings <$> arbitraryTransaction <*> arbitraryWarnings
    , Quarantined <$> arbitraryTransaction <*> arbitraryQuarantineReasons
    , SkippedDuplicate <$> arbitrarySkippedDuplicate
    , Rejected <$> arbitraryRejectedRow
    , Fatal <$> arbitraryFatalError
    ]

The shape of this generator *mirrors* the shape of the type. Six
constructors in the type, six branches in the generator. When you
add a seventh constructor (chapter 7's hypothetical
`DeferredForManualReview`), this generator becomes incomplete — and
the existing properties keep passing only on the cases they cover,
no longer exercising the new one. You don't get false confidence;
you get pressure to add the seventh branch to the generator too.

Here's a third property that demonstrates the "laws as constructor
properties" point:

In [ ]:
prop "fatal row decisions block the batch" \(NonEmptyFatalContexts contexts) ->
  case batchOutcome (classifyBatch defaultConfig Normal contexts) of
    BatchBlockedByFatal _ -> property True
    BatchUploadable -> counterexample "expected fatal to block batch" False

Read: for any batch of contexts that contains at least one fatal
row, the batch outcome must be `BatchBlockedByFatal`. The
`NonEmptyFatalContexts` generator always inserts a fatal context
somewhere in the batch, then asserts the outcome.

This isn't testing one example. It's testing the rule. Domain rule 7
from CLAUDE.md ("fatal errors block the entire batch") *is* this
property. The test is the rule's executable statement.

## What the type system covers vs. what the tests cover

<div class="mermaid-diagram" style="width:100%">
<svg id="mermaid-figure-1" xmlns="http://www.w3.org/2000/svg" xlink="http://www.w3.org/1999/xlink" class="flowchart" viewbox="0 0 346 1373" role="graphics-document document" aria-roledescription="flowchart-v2" style="width:100%;height:auto;max-width:100%;display:block">
<style>#mermaid-figure-1{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;fill:#000000;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#mermaid-figure-1 .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#mermaid-figure-1 .error-icon{fill:#552222;}#mermaid-figure-1 .error-text{fill:#552222;stroke:#552222;}#mermaid-figure-1 .edge-thickness-normal{stroke-width:1px;}#mermaid-figure-1 .edge-thickness-thick{stroke-width:3.5px;}#mermaid-figure-1 .edge-pattern-solid{stroke-dasharray:0;}#mermaid-figure-1 .edge-thickness-invisible{stroke-width:0;fill:none;}#mermaid-figure-1 .edge-pattern-dashed{stroke-dasharray:3;}#mermaid-figure-1 .edge-pattern-dotted{stroke-dasharray:2;}#mermaid-figure-1 .marker{fill:#666;stroke:#666;}#mermaid-figure-1 .marker.cross{stroke:#666;}#mermaid-figure-1 svg{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;}#mermaid-figure-1 p{margin:0;}#mermaid-figure-1 .label{font-family:"trebuchet ms",verdana,arial,sans-serif;color:#000000;}#mermaid-figure-1 .cluster-label text{fill:#333;}#mermaid-figure-1 .cluster-label span{color:#333;}#mermaid-figure-1 .cluster-label span p{background-color:transparent;}#mermaid-figure-1 .label text,#mermaid-figure-1 span{fill:#000000;color:#000000;}#mermaid-figure-1 .node rect,#mermaid-figure-1 .node circle,#mermaid-figure-1 .node ellipse,#mermaid-figure-1 .node polygon,#mermaid-figure-1 .node path{fill:#eee;stroke:#999;stroke-width:1px;}#mermaid-figure-1 .rough-node .label text,#mermaid-figure-1 .node .label text,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-anchor:middle;}#mermaid-figure-1 .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#mermaid-figure-1 .rough-node .label,#mermaid-figure-1 .node .label,#mermaid-figure-1 .image-shape .label,#mermaid-figure-1 .icon-shape .label{text-align:center;}#mermaid-figure-1 .node.clickable{cursor:pointer;}#mermaid-figure-1 .root .anchor path{fill:#666!important;stroke-width:0;stroke:#666;}#mermaid-figure-1 .arrowheadPath{fill:#333333;}#mermaid-figure-1 .edgePath .path{stroke:#666;stroke-width:2.0px;}#mermaid-figure-1 .flowchart-link{stroke:#666;fill:none;}#mermaid-figure-1 .edgeLabel{background-color:white;text-align:center;}#mermaid-figure-1 .edgeLabel p{background-color:white;}#mermaid-figure-1 .edgeLabel rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .labelBkg{background-color:rgba(255, 255, 255, 0.5);}#mermaid-figure-1 .cluster rect{fill:hsl(0, 0%, 98.9215686275%);stroke:#707070;stroke-width:1px;}#mermaid-figure-1 .cluster text{fill:#333;}#mermaid-figure-1 .cluster span{color:#333;}#mermaid-figure-1 div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:12px;background:hsl(-160, 0%, 93.3333333333%);border:1px solid #707070;border-radius:2px;pointer-events:none;z-index:100;}#mermaid-figure-1 .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#000000;}#mermaid-figure-1 rect.text{fill:none;stroke-width:0;}#mermaid-figure-1 .icon-shape,#mermaid-figure-1 .image-shape{background-color:white;text-align:center;}#mermaid-figure-1 .icon-shape p,#mermaid-figure-1 .image-shape p{background-color:white;padding:2px;}#mermaid-figure-1 .icon-shape rect,#mermaid-figure-1 .image-shape rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-1 .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#mermaid-figure-1 .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#mermaid-figure-1 :root{--mermaid-font-family:"trebuchet ms",verdana,arial,sans-serif;}</style>
<g><marker id="mermaid-1779381581042_flowchart-v2-pointEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381581042_flowchart-v2-pointStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="4.5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381581042_flowchart-v2-circleEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="11" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381581042_flowchart-v2-circleStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="-1" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381581042_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="12" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381581042_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="-1" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"></g><g class="edgeLabels"></g><g class="nodes"><g class="root" transform="translate(0, 0)"><g class="clusters"><g class="cluster" id="HS" data-look="classic"><rect style="" x="8" y="8" width="330" height="435"></rect><g class="cluster-label" transform="translate(146.765625, 8)"><foreignobject width="52.46875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Haskell
</p>
</span>
</div>
</foreignobject></g></g></g><g class="edgePaths"><path d="M173,147.5L173,153.75C173,160,173,172.5,173,184.333C173,196.167,173,207.333,173,212.917L173,218.5" id="L_HST_HSP_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_HST_HSP_0" data-points="W3sieCI6MTczLCJ5IjoxNDcuNX0seyJ4IjoxNzMsInkiOjE4NX0seyJ4IjoxNzMsInkiOjIyMi41fV0=" marker-end="url(#mermaid-1779381581042_flowchart-v2-pointEnd)"></path><path d="M173,276.5L173,282.75C173,289,173,301.5,173,313.333C173,325.167,173,336.333,173,341.917L173,347.5" id="L_HSP_HSR_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_HSP_HSR_0" data-points="W3sieCI6MTczLCJ5IjoyNzYuNX0seyJ4IjoxNzMsInkiOjMxNH0seyJ4IjoxNzMsInkiOjM1MS41fV0=" marker-end="url(#mermaid-1779381581042_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel"><g class="label" data-id="L_HST_HSP_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_HSP_HSR_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g></g><g class="nodes"><g class="node default" id="flowchart-HST-12" transform="translate(173, 96.5)"><rect class="basic label-container" style="" x="-130" y="-51" width="260" height="102"></rect><g class="label" style="" transform="translate(-100, -36)"><rect></rect><foreignobject width="200" height="72">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
Types: ~90%<br>(above + NonEmpty + IO purity)
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-HSP-13" transform="translate(173, 249.5)"><rect class="basic label-container" style="" x="-100.03125" y="-27" width="200.0625" height="54"></rect><g class="label" style="" transform="translate(-70.03125, -12)"><rect></rect><foreignobject width="140.0625" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Property tests: ~9%
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-HSR-14" transform="translate(173, 378.5)"><rect class="basic label-container" style="" x="-80.46875" y="-27" width="160.9375" height="54"></rect><g class="label" style="" transform="translate(-50.46875, -12)"><rect></rect><foreignobject width="100.9375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Runtime: ~1%
</p>
</span>
</div>
</foreignobject></g></g></g></g><g class="root" transform="translate(0, 485)"><g class="clusters"><g class="cluster" id="FS" data-look="classic"><rect style="" x="8" y="8" width="330" height="435"></rect><g class="cluster-label" transform="translate(163.6640625, 8)"><foreignobject width="18.671875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
F#
</p>
</span>
</div>
</foreignobject></g></g></g><g class="edgePaths"><path d="M173,147.5L173,153.75C173,160,173,172.5,173,184.333C173,196.167,173,207.333,173,212.917L173,218.5" id="L_FST_FSP_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_FST_FSP_0" data-points="W3sieCI6MTczLCJ5IjoxNDcuNX0seyJ4IjoxNzMsInkiOjE4NX0seyJ4IjoxNzMsInkiOjIyMi41fV0=" marker-end="url(#mermaid-1779381581042_flowchart-v2-pointEnd)"></path><path d="M173,276.5L173,282.75C173,289,173,301.5,173,313.333C173,325.167,173,336.333,173,341.917L173,347.5" id="L_FSP_FSR_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_FSP_FSR_0" data-points="W3sieCI6MTczLCJ5IjoyNzYuNX0seyJ4IjoxNzMsInkiOjMxNH0seyJ4IjoxNzMsInkiOjM1MS41fV0=" marker-end="url(#mermaid-1779381581042_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel"><g class="label" data-id="L_FST_FSP_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_FSP_FSR_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g></g><g class="nodes"><g class="node default" id="flowchart-FST-6" transform="translate(173, 96.5)"><rect class="basic label-container" style="" x="-130" y="-51" width="260" height="102"></rect><g class="label" style="" transform="translate(-100, -36)"><rect></rect><foreignobject width="200" height="72">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
Types: ~70%<br>(sum types, exhaustiveness, no null)
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-FSP-7" transform="translate(173, 249.5)"><rect class="basic label-container" style="" x="-73.8046875" y="-27" width="147.609375" height="54"></rect><g class="label" style="" transform="translate(-43.8046875, -12)"><rect></rect><foreignobject width="87.609375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Tests: ~20%
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-FSR-8" transform="translate(173, 378.5)"><rect class="basic label-container" style="" x="-84.921875" y="-27" width="169.84375" height="54"></rect><g class="label" style="" transform="translate(-54.921875, -12)"><rect></rect><foreignobject width="109.84375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Runtime: ~10%
</p>
</span>
</div>
</foreignobject></g></g></g></g><g class="root" transform="translate(39.0859375, 970)"><g class="clusters"><g class="cluster" id="CS" data-look="classic"><rect style="" x="8" y="8" width="251.828125" height="387"></rect><g class="cluster-label" transform="translate(95.6796875, 8)"><foreignobject width="76.46875" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Normal C#
</p>
</span>
</div>
</foreignobject></g></g></g><g class="edgePaths"><path d="M133.914,99.5L133.914,105.75C133.914,112,133.914,124.5,133.914,136.333C133.914,148.167,133.914,159.333,133.914,164.917L133.914,170.5" id="L_CST_CSP_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_CST_CSP_0" data-points="W3sieCI6MTMzLjkxNDA2MjUsInkiOjk5LjV9LHsieCI6MTMzLjkxNDA2MjUsInkiOjEzN30seyJ4IjoxMzMuOTE0MDYyNSwieSI6MTc0LjV9XQ==" marker-end="url(#mermaid-1779381581042_flowchart-v2-pointEnd)"></path><path d="M133.914,228.5L133.914,234.75C133.914,241,133.914,253.5,133.914,265.333C133.914,277.167,133.914,288.333,133.914,293.917L133.914,299.5" id="L_CSP_CSR_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_CSP_CSR_0" data-points="W3sieCI6MTMzLjkxNDA2MjUsInkiOjIyOC41fSx7IngiOjEzMy45MTQwNjI1LCJ5IjoyNjZ9LHsieCI6MTMzLjkxNDA2MjUsInkiOjMwMy41fV0=" marker-end="url(#mermaid-1779381581042_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel"><g class="label" data-id="L_CST_CSP_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_CSP_CSR_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g></g><g class="nodes"><g class="node default" id="flowchart-CST-0" transform="translate(133.9140625, 72.5)"><rect class="basic label-container" style="" x="-76.4765625" y="-27" width="152.953125" height="54"></rect><g class="label" style="" transform="translate(-46.4765625, -12)"><rect></rect><foreignobject width="92.953125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Types: ~10%
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-CSP-1" transform="translate(133.9140625, 201.5)"><rect class="basic label-container" style="" x="-73.8046875" y="-27" width="147.609375" height="54"></rect><g class="label" style="" transform="translate(-43.8046875, -12)"><rect></rect><foreignobject width="87.609375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Tests: ~30%
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-CSR-2" transform="translate(133.9140625, 330.5)"><rect class="basic label-container" style="" x="-90.9140625" y="-27" width="181.828125" height="54"></rect><g class="label" style="" transform="translate(-60.9140625, -12)"><rect></rect><foreignobject width="121.828125" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Runtime: the rest
</p>
</span>
</div>
</foreignobject></g></g></g></g></g></g></g>
</svg>
</div>

Percentages are rough. The point is the *shift in burden*. In normal
C# the runtime is the last-line defender. In Haskell the runtime
barely catches anything because the type system caught it and (where
types can't reach) properties caught it.

## Is it easier or harder to read?

Honest answer: harder up front, easier once you can.

Steeper learning curve than F# (already steeper than C#). The primer
in chapter 7 is enough to read the domain code, but Haskell in the
wild uses applicative combinators (`<$>`, `<*>`, `>>=`), monad
transformers, type-class machinery, GADT syntax — none of which
appear in this codebase, all of which a real-world Haskell project
might. The vocabulary is large.

Once you can read it, the domain is more legible than at any of the
lower rungs. The normal-C# `RowDecision` from chapter 2 is nine
fields, four nullable, three mutable lists, six valid status strings
encoded in a comment — the shape doesn't tell you which combinations
are legal. The Haskell `RowDecision` is eight lines and every
constructor carries exactly what it needs, no more, no less. The
shape *is* the documentation.

## How easy is it to extend?

Adding a seventh `RowDecision` case, say `DeferredForManualReview`:

In [ ]:
data RowDecision
  = Accepted FuelTransaction
  | AcceptedWithWarnings FuelTransaction (NonEmpty ValidationWarning)
  | Quarantined FuelTransaction (NonEmpty QuarantineReason)
  | SkippedDuplicate SkippedDuplicate
  | Rejected RejectedRow
  | Fatal FatalError
  | DeferredForManualReview FuelTransaction String   -- NEW
  deriving stock (Eq, Show)

One line. Now run `cabal build`. Every `case` in the codebase that
matches on `RowDecision` emits a `-Wincomplete-uni-patterns`
warning. The compiler enumerates them:

- `Summary.hs`'s `addDecision` and `collectFatal`
- `Api.hs`'s `toDecisionDto`
- `Audit.hs`'s `projectRow` and `acceptedTransactionIds`
- `Report.hs`'s `acceptedTransactionIds`, `rejectedRowNumbers`,
  `quarantinedRows`, `skippedDuplicateRowNumbers`
- `DecisionEngine.hs` (only if a `case` reads `RowDecision` —
  here it produces them, so it's fine)

With `-Werror` the build fails until each is handled. You **cannot
ship a half-extended codebase**. The compiler has located every
consumer; you fix them; you ship.

Property tests are the second layer. The `arbitraryDecision`
generator covers six branches today. After the change, it still
covers six branches and silently ignores the seventh — the
properties still pass, but on a narrower domain. The generator
itself isn't compile-checked against the constructors (there's no
"exhaustive generator" feature), so this is a soft signal rather
than a build break. Catching it is part of the "extend the generator
when you extend the type" discipline.

<div class="mermaid-diagram" style="width:100%">
<svg id="mermaid-figure-2" xmlns="http://www.w3.org/2000/svg" xlink="http://www.w3.org/1999/xlink" class="flowchart" viewbox="0 0 1708.609375 118" role="graphics-document document" aria-roledescription="flowchart-v2" style="width:100%;height:auto;max-width:100%;display:block">
<style>#mermaid-figure-2{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;fill:#000000;}@keyframes edge-animation-frame{from{stroke-dashoffset:0;}}@keyframes dash{to{stroke-dashoffset:0;}}#mermaid-figure-2 .edge-animation-slow{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 50s linear infinite;stroke-linecap:round;}#mermaid-figure-2 .edge-animation-fast{stroke-dasharray:9,5!important;stroke-dashoffset:900;animation:dash 20s linear infinite;stroke-linecap:round;}#mermaid-figure-2 .error-icon{fill:#552222;}#mermaid-figure-2 .error-text{fill:#552222;stroke:#552222;}#mermaid-figure-2 .edge-thickness-normal{stroke-width:1px;}#mermaid-figure-2 .edge-thickness-thick{stroke-width:3.5px;}#mermaid-figure-2 .edge-pattern-solid{stroke-dasharray:0;}#mermaid-figure-2 .edge-thickness-invisible{stroke-width:0;fill:none;}#mermaid-figure-2 .edge-pattern-dashed{stroke-dasharray:3;}#mermaid-figure-2 .edge-pattern-dotted{stroke-dasharray:2;}#mermaid-figure-2 .marker{fill:#666;stroke:#666;}#mermaid-figure-2 .marker.cross{stroke:#666;}#mermaid-figure-2 svg{font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:16px;}#mermaid-figure-2 p{margin:0;}#mermaid-figure-2 .label{font-family:"trebuchet ms",verdana,arial,sans-serif;color:#000000;}#mermaid-figure-2 .cluster-label text{fill:#333;}#mermaid-figure-2 .cluster-label span{color:#333;}#mermaid-figure-2 .cluster-label span p{background-color:transparent;}#mermaid-figure-2 .label text,#mermaid-figure-2 span{fill:#000000;color:#000000;}#mermaid-figure-2 .node rect,#mermaid-figure-2 .node circle,#mermaid-figure-2 .node ellipse,#mermaid-figure-2 .node polygon,#mermaid-figure-2 .node path{fill:#eee;stroke:#999;stroke-width:1px;}#mermaid-figure-2 .rough-node .label text,#mermaid-figure-2 .node .label text,#mermaid-figure-2 .image-shape .label,#mermaid-figure-2 .icon-shape .label{text-anchor:middle;}#mermaid-figure-2 .node .katex path{fill:#000;stroke:#000;stroke-width:1px;}#mermaid-figure-2 .rough-node .label,#mermaid-figure-2 .node .label,#mermaid-figure-2 .image-shape .label,#mermaid-figure-2 .icon-shape .label{text-align:center;}#mermaid-figure-2 .node.clickable{cursor:pointer;}#mermaid-figure-2 .root .anchor path{fill:#666!important;stroke-width:0;stroke:#666;}#mermaid-figure-2 .arrowheadPath{fill:#333333;}#mermaid-figure-2 .edgePath .path{stroke:#666;stroke-width:2.0px;}#mermaid-figure-2 .flowchart-link{stroke:#666;fill:none;}#mermaid-figure-2 .edgeLabel{background-color:white;text-align:center;}#mermaid-figure-2 .edgeLabel p{background-color:white;}#mermaid-figure-2 .edgeLabel rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-2 .labelBkg{background-color:rgba(255, 255, 255, 0.5);}#mermaid-figure-2 .cluster rect{fill:hsl(0, 0%, 98.9215686275%);stroke:#707070;stroke-width:1px;}#mermaid-figure-2 .cluster text{fill:#333;}#mermaid-figure-2 .cluster span{color:#333;}#mermaid-figure-2 div.mermaidTooltip{position:absolute;text-align:center;max-width:200px;padding:2px;font-family:"trebuchet ms",verdana,arial,sans-serif;font-size:12px;background:hsl(-160, 0%, 93.3333333333%);border:1px solid #707070;border-radius:2px;pointer-events:none;z-index:100;}#mermaid-figure-2 .flowchartTitleText{text-anchor:middle;font-size:18px;fill:#000000;}#mermaid-figure-2 rect.text{fill:none;stroke-width:0;}#mermaid-figure-2 .icon-shape,#mermaid-figure-2 .image-shape{background-color:white;text-align:center;}#mermaid-figure-2 .icon-shape p,#mermaid-figure-2 .image-shape p{background-color:white;padding:2px;}#mermaid-figure-2 .icon-shape rect,#mermaid-figure-2 .image-shape rect{opacity:0.5;background-color:white;fill:white;}#mermaid-figure-2 .label-icon{display:inline-block;height:1em;overflow:visible;vertical-align:-0.125em;}#mermaid-figure-2 .node .label-icon path{fill:currentColor;stroke:revert;stroke-width:revert;}#mermaid-figure-2 :root{--mermaid-font-family:"trebuchet ms",verdana,arial,sans-serif;}</style>
<g><marker id="mermaid-1779381581218_flowchart-v2-pointEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 0 L 10 5 L 0 10 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381581218_flowchart-v2-pointStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="4.5" refy="5" markerunits="userSpaceOnUse" markerwidth="8" markerheight="8" orient="auto"><path d="M 0 5 L 10 10 L 10 0 z" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381581218_flowchart-v2-circleEnd" class="marker flowchart-v2" viewbox="0 0 10 10" refx="11" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381581218_flowchart-v2-circleStart" class="marker flowchart-v2" viewbox="0 0 10 10" refx="-1" refy="5" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><circle cx="5" cy="5" r="5" class="arrowMarkerPath" style="stroke-width: 1; stroke-dasharray: 1, 0;"></circle></marker><marker id="mermaid-1779381581218_flowchart-v2-crossEnd" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="12" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><marker id="mermaid-1779381581218_flowchart-v2-crossStart" class="marker cross flowchart-v2" viewbox="0 0 11 11" refx="-1" refy="5.2" markerunits="userSpaceOnUse" markerwidth="11" markerheight="11" orient="auto"><path d="M 1,1 l 9,9 M 10,1 l -9,9" class="arrowMarkerPath" style="stroke-width: 2; stroke-dasharray: 1, 0;"></path></marker><g class="root"><g class="clusters"></g><g class="edgePaths"><path d="M268,59L272.167,59C276.333,59,284.667,59,292.333,59C300,59,307,59,310.5,59L314,59" id="L_ADD_BUILD_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_ADD_BUILD_0" data-points="W3sieCI6MjY4LCJ5Ijo1OX0seyJ4IjoyOTMsInkiOjU5fSx7IngiOjMxOCwieSI6NTl9XQ==" marker-end="url(#mermaid-1779381581218_flowchart-v2-pointEnd)"></path><path d="M545.766,59L549.932,59C554.099,59,562.432,59,570.099,59C577.766,59,584.766,59,588.266,59L591.766,59" id="L_BUILD_WARN_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_BUILD_WARN_0" data-points="W3sieCI6NTQ1Ljc2NTYyNSwieSI6NTl9LHsieCI6NTcwLjc2NTYyNSwieSI6NTl9LHsieCI6NTk1Ljc2NTYyNSwieSI6NTl9XQ==" marker-end="url(#mermaid-1779381581218_flowchart-v2-pointEnd)"></path><path d="M855.766,59L859.932,59C864.099,59,872.432,59,880.099,59C887.766,59,894.766,59,898.266,59L901.766,59" id="L_WARN_FIX_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_WARN_FIX_0" data-points="W3sieCI6ODU1Ljc2NTYyNSwieSI6NTl9LHsieCI6ODgwLjc2NTYyNSwieSI6NTl9LHsieCI6OTA1Ljc2NTYyNSwieSI6NTl9XQ==" marker-end="url(#mermaid-1779381581218_flowchart-v2-pointEnd)"></path><path d="M1112.5,59L1116.667,59C1120.833,59,1129.167,59,1136.833,59C1144.5,59,1151.5,59,1155,59L1158.5,59" id="L_FIX_PROPS_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_FIX_PROPS_0" data-points="W3sieCI6MTExMi41LCJ5Ijo1OX0seyJ4IjoxMTM3LjUsInkiOjU5fSx7IngiOjExNjIuNSwieSI6NTl9XQ==" marker-end="url(#mermaid-1779381581218_flowchart-v2-pointEnd)"></path><path d="M1422.5,59L1426.667,59C1430.833,59,1439.167,59,1446.833,59C1454.5,59,1461.5,59,1465,59L1468.5,59" id="L_PROPS_PASS_0" class="edge-thickness-normal edge-pattern-solid edge-thickness-normal edge-pattern-solid flowchart-link" style=";" data-edge="true" data-et="edge" data-id="L_PROPS_PASS_0" data-points="W3sieCI6MTQyMi41LCJ5Ijo1OX0seyJ4IjoxNDQ3LjUsInkiOjU5fSx7IngiOjE0NzIuNSwieSI6NTl9XQ==" marker-end="url(#mermaid-1779381581218_flowchart-v2-pointEnd)"></path></g><g class="edgeLabels"><g class="edgeLabel"><g class="label" data-id="L_ADD_BUILD_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_BUILD_WARN_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_WARN_FIX_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_FIX_PROPS_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g><g class="edgeLabel"><g class="label" data-id="L_PROPS_PASS_0" transform="translate(0, 0)"><foreignobject width="0" height="0">
<div class="labelBkg" data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="edgeLabel"></span>
</div>
</foreignobject></g></g></g><g class="nodes"><g class="node default" id="flowchart-ADD-0" transform="translate(138, 59)"><rect class="basic label-container" style="" x="-130" y="-51" width="260" height="102"></rect><g class="label" style="" transform="translate(-100, -36)"><rect></rect><foreignobject width="200" height="72">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
Add DeferredForManualReview case
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-BUILD-2" transform="translate(431.8828125, 59)"><rect class="basic label-container" style="" x="-113.8828125" y="-27" width="227.765625" height="54"></rect><g class="label" style="" transform="translate(-83.8828125, -12)"><rect></rect><foreignobject width="167.765625" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
cabal build with -Werror
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-WARN-4" transform="translate(725.765625, 59)"><rect class="basic label-container" style="" x="-130" y="-39" width="260" height="78"></rect><g class="label" style="" transform="translate(-100, -24)"><rect></rect><foreignobject width="200" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
Compiler enumerates ~10 consumer sites
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-FIX-6" transform="translate(1009.1328125, 59)"><rect class="basic label-container" style="" x="-103.3671875" y="-27" width="206.734375" height="54"></rect><g class="label" style="" transform="translate(-73.3671875, -12)"><rect></rect><foreignobject width="146.734375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Fix each case match
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-PROPS-8" transform="translate(1292.5, 59)"><rect class="basic label-container" style="" x="-130" y="-39" width="260" height="78"></rect><g class="label" style="" transform="translate(-100, -24)"><rect></rect><foreignobject width="200" height="48">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table; white-space: break-spaces; line-height: 1.5; max-width: 200px; text-align: center; width: 200px;">
<span class="nodeLabel">
<p>
Update arbitraryDecision generator
</p>
</span>
</div>
</foreignobject></g></g><g class="node default" id="flowchart-PASS-10" transform="translate(1586.5546875, 59)"><rect class="basic label-container" style="" x="-114.0546875" y="-27" width="228.109375" height="54"></rect><g class="label" style="" transform="translate(-84.0546875, -12)"><rect></rect><foreignobject width="168.109375" height="24">
<div data-xmlns="http://www.w3.org/1999/xhtml" style="display: table-cell; white-space: nowrap; line-height: 1.5; max-width: 200px; text-align: center;">
<span class="nodeLabel">
<p>
Build green, props pass
</p>
</span>
</div>
</foreignobject></g></g></g></g></g>
</svg>
</div>

Compare to normal C# where the same change is "add the string
'DeferredForManualReview' and hope nothing downstream needs to know."
Five hops above the cliff base.

## Where this still leaks

Haskell stayed the strongest reference model in the V3 scoring
(`docs/v3-results.md`). The domain is tight, the property tests are
real, the boundary errors are typed. The score it lost was on a
single dimension: **practical fit for a .NET shop**.

From the V3 results:

> The API boundary is an in-process typed facade rather than a
> realistic service adapter; repository errors lose details when
> converted to fatal decisions; vehicle lookup lacks the
> ambiguous-vehicle case present in the other v3 implementations.
> No real serialization or .NET-friendly adapter shape emerged.

The Haskell `Api.hs` is impeccable as a type-driven mapping
exercise. It parses CSV-shaped strings, accumulates typed mapping
errors, and emits typed DTOs. What it isn't, is a service that fits
naturally into a C#/.NET application's runtime — there's no
ASP.NET hosting model, no `System.Text.Json` serialisation, no
DI-shaped repository client.

That's the boundary tradeoff the next part picks up. The
type system on the cliff face is everything we said it was. The
adapter shape that brings it into a real .NET application is — for
this organisation — a separate problem, and one we revisit in [Part Ib
chapter 11](../part1b/11-boundary-returns.ipynb).

For Part I's narrative, though, the Haskell column is the answer to
the central question: **what kind of bug can the type system
make unrepresentable?** Seven out of seven. The cliff is real, and
once you're standing on it, the floor is a long way down.

[next →](09-rust-lands.ipynb)